In [1]:
!pip install -q pandas numpy requests beautifulsoup4 tqdm lxml

In [2]:
import pandas as pd
import numpy as np
import requests
import re
import time

from bs4 import BeautifulSoup
from datetime import datetime
from tqdm import tqdm

In [3]:
OUTPUT_FILE = "cinema-scrape.csv"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

SOURCE_URLS = [
    {
        "url": "https://blog.ultravoucher.co.id/harga-tiket-bioskop/",
        "source_platform": "UltraVoucher",
        "source_type": "cinema_ticket",
        "raw_category": "cinema ticket",
        "city": "Jakarta",
        "keyword": "harga tiket bioskop"
    },
    {
        "url": "https://nonabelanja.com/menu-cgv/",
        "source_platform": "NonaBelanja",
        "source_type": "cinema_concession",
        "raw_category": "cinema snack / drink",
        "city": "unknown",
        "keyword": "menu cgv"
    }
]

REQUEST_DELAY = 1

In [4]:
def clean_text(text):
    if text is None:
        return ""
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def clean_price(text):
    if text is None or pd.isna(text):
        return np.nan

    text = str(text)
    text = re.sub(r"[^0-9]", "", text)

    if text == "":
        return np.nan

    return int(text)


def extract_prices(text):
    text = clean_text(text)
    matches = re.findall(r"Rp\s*[0-9][0-9\.\,]*|[0-9]{1,3}(?:\.[0-9]{3})+", text)
    prices = []

    for match in matches:
        price = clean_price(match)

        if not pd.isna(price):
            prices.append(price)

    return prices


def is_valid_item_name(text):
    text = clean_text(text)
    lower = text.lower()

    if len(text) < 3:
        return False

    if len(text) > 150:
        return False

    if lower.startswith("rp"):
        return False

    if re.fullmatch(r"[0-9\.\,\s]+", text):
        return False

    noise_words = [
        "harga", "daftar", "update", "terbaru", "artikel",
        "blog", "voucher", "ultravoucher", "nonabelanja",
        "baca juga", "halaman", "lihat", "berikut",
        "di bawah ini", "adalah", "anda bisa", "promo",
        "diskon", "review", "rating", "terms", "privacy",
        "comment", "share", "facebook", "twitter",
        "whatsapp", "instagram"
    ]

    if any(word in lower for word in noise_words):
        return False

    return True


def infer_city(text):
    text = str(text).lower()

    cities = [
        "jakarta", "semarang", "bandung", "yogyakarta", "surabaya",
        "medan", "tangerang", "bekasi", "bogor", "depok",
        "malang", "solo", "denpasar", "balikpapan", "makassar"
    ]

    for city in cities:
        if city in text:
            return city.title()

    return "Unknown"


def infer_raw_category(text):
    text = str(text).lower()

    if any(k in text for k in ["popcorn", "french fries", "hotdog", "siomay", "sausage", "sampler"]):
        return "movie snack"

    if any(k in text for k in ["soft drink", "milo", "lemon tea", "tea", "soda", "drink"]):
        return "movie drink"

    if any(k in text for k in ["premiere", "imax", "satin", "sweetbox", "regular", "weekday", "weekend", "holiday", "jumat", "tiket"]):
        return "cinema ticket"

    return "cinema"

In [5]:
def get_page_title(soup):
    h1 = soup.find("h1")

    if h1:
        return clean_text(h1.get_text())

    title = soup.find("title")

    if title:
        return clean_text(title.get_text())

    return "Unknown Source"


def extract_from_tables(soup, source):
    rows = []

    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            cells = [clean_text(td.get_text(" ")) for td in tr.find_all(["td", "th"])]

            if len(cells) < 2:
                continue

            row_text = " ".join(cells)
            prices = extract_prices(row_text)

            if not prices:
                continue

            name_parts = []

            for cell in cells:
                if not extract_prices(cell):
                    if is_valid_item_name(cell):
                        name_parts.append(cell)

            if not name_parts:
                name_candidate = re.sub(r"Rp\s*[0-9][0-9\.\,]*", "", row_text)
                name_candidate = clean_text(name_candidate)
            else:
                name_candidate = " - ".join(name_parts)

            if not is_valid_item_name(name_candidate):
                continue

            for price in prices:
                rows.append({
                    "item_name": name_candidate,
                    "price": price,
                    "section": None,
                    "extract_method": "html_table"
                })

    return rows


def extract_from_text(soup, source):
    text = soup.get_text("\n")
    lines = [clean_text(line) for line in text.splitlines()]
    lines = [line for line in lines if line]

    rows = []
    current_section = None

    section_keywords = [
        "xxi", "cinema xxi", "cgv", "cinepolis",
        "harga tiket", "menu", "popcorn", "snack", "drink"
    ]

    for idx, line in enumerate(lines):
        lower = line.lower()

        if any(k in lower for k in section_keywords) and not extract_prices(line):
            current_section = line
            continue

        prices = extract_prices(line)

        if not prices:
            continue

        name_candidate = re.sub(r"Rp\s*[0-9][0-9\.\,]*", "", line)
        name_candidate = clean_text(name_candidate)

        if not is_valid_item_name(name_candidate):
            candidates = []

            if idx - 1 >= 0:
                candidates.append(lines[idx - 1])

            if idx - 2 >= 0:
                candidates.append(lines[idx - 2])

            for candidate in candidates:
                if is_valid_item_name(candidate):
                    name_candidate = candidate
                    break

        if not is_valid_item_name(name_candidate):
            continue

        for price in prices:
            rows.append({
                "item_name": name_candidate,
                "price": price,
                "section": current_section,
                "extract_method": "text_pattern"
            })

    return rows


def deduplicate_items(items):
    unique = []
    seen = set()

    for item in items:
        if pd.isna(item["price"]):
            continue

        key = (
            str(item.get("section")).lower(),
            str(item["item_name"]).lower(),
            int(item["price"])
        )

        if key not in seen:
            seen.add(key)
            unique.append(item)

    return unique

In [6]:
def scrape_source(source):
    rows = []
    url = source["url"]

    try:
        response = requests.get(url, headers=HEADERS, timeout=30)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "lxml")
        page_title = get_page_title(soup)

        items = []
        items.extend(extract_from_tables(soup, source))
        items.extend(extract_from_text(soup, source))
        items = deduplicate_items(items)

        for item in items:
            item_text = " ".join([
                str(source["raw_category"]),
                str(item.get("section")),
                str(item["item_name"])
            ])

            rows.append({
                "source_platform": source["source_platform"],
                "source_type": source["source_type"],
                "restaurant_name": page_title,
                "city": source["city"] if source["city"] != "unknown" else infer_city(page_title + " " + url),
                "raw_category": infer_raw_category(item_text),
                "section": item.get("section"),
                "menu_name": item["item_name"],
                "price": item["price"],
                "source_url": url,
                "search_keyword": source["keyword"],
                "extract_method": item["extract_method"],
                "scraped_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            })

    except Exception as e:
        print(f"Failed: {url} | {e}")

    return rows

In [7]:
all_rows = []

for source in tqdm(SOURCE_URLS, desc="Scraping cinema sources"):
    rows = scrape_source(source)
    all_rows.extend(rows)

    if all_rows:
        pd.DataFrame(all_rows).to_csv("checkpoint_cinema_scrape.csv", index=False)

    time.sleep(REQUEST_DELAY)

df_cinema = pd.DataFrame(all_rows)

if df_cinema.empty:
    print("Tidak ada data berhasil discrape.")
else:
    df_cinema = df_cinema.drop_duplicates(
        subset=["source_platform", "menu_name", "price", "source_url"],
        keep="first"
    ).reset_index(drop=True)

    df_cinema["price"] = pd.to_numeric(df_cinema["price"], errors="coerce")

    df_cinema = df_cinema[
        (df_cinema["price"].isna()) |
        ((df_cinema["price"] >= 3000) & (df_cinema["price"] <= 500000))
    ].copy()

    df_cinema = df_cinema.reset_index(drop=True)

    df_cinema.to_csv(OUTPUT_FILE, index=False)

    print("=" * 80)
    print("SCRAPING CINEMA / BIOSKOP SELESAI")
    print("=" * 80)
    print(f"Output file        : {OUTPUT_FILE}")
    print(f"Total rows         : {len(df_cinema)}")
    print(f"Source platform    : {df_cinema['source_platform'].unique().tolist()}")
    print(f"Raw categories     : {df_cinema['raw_category'].unique().tolist()}")
    print(f"Min price          : {df_cinema['price'].min()}")
    print(f"Median price       : {df_cinema['price'].median()}")
    print(f"Max price          : {df_cinema['price'].max()}")

    display(df_cinema.head(30))

Scraping cinema sources: 100%|██████████| 2/2 [00:07<00:00,  3.58s/it]

SCRAPING CINEMA / BIOSKOP SELESAI
Output file        : cinema-scrape.csv
Total rows         : 343
Source platform    : ['UltraVoucher', 'NonaBelanja']
Raw categories     : ['cinema', 'cinema ticket', 'movie snack', 'movie drink']
Min price          : 25000
Median price       : 50000.0
Max price          : 300000


,source_platform,source_type,restaurant_name,city,raw_category,section,menu_name,price,source_url,search_keyword,extract_method,scraped_at
0,UltraVoucher,cinema_ticket,Harga Tiket Bioskop XXI dan CGV Lengkap di Jak...,Jakarta,cinema,None,KALIBATA XXI - Tersedia - Tidak Tersedia - Tid...,40000,https://blog.ultravoucher.co.id/harga-tiket-bi...,harga tiket bioskop,html_table,2026-04-28 03:19:59
1,UltraVoucher,cinema_ticket,Harga Tiket Bioskop XXI dan CGV Lengkap di Jak...,Jakarta,cinema,None,KALIBATA XXI - Tersedia - Tidak Tersedia - Tid...,35000,https://blog.ultravoucher.co.id/harga-tiket-bi...,harga tiket bioskop,html_table,2026-04-28 03:19:59
2,UltraVoucher,cinema_ticket,Harga Tiket Bioskop XXI dan CGV Lengkap di Jak...,Jakarta,cinema,None,KALIBATA XXI - Tersedia - Tidak Tersedia - Tid...,50000,https://blog.ultravoucher.co.id/harga-tiket-bi...,harga tiket bioskop,html_table,2026-04-28 03:19:59
3,UltraVoucher,cinema_ticket,Harga Tiket Bioskop XXI dan CGV Lengkap di Jak...,Jakarta,cinema,None,KOTA KASABLANKA XXI - Tersedia - Tersedia - Ti...,60000,https://blog.ultravoucher.co.id/harga-tiket-bi...,harga tiket bioskop,html_table,2026-04-28 03:19:59
4,UltraVoucher,cinema_ticket,Harga Tiket Bioskop XXI dan CGV Lengkap di Jak...,Jakarta,cinema,None,KOTA KASABLANKA XXI - Tersedia - Tersedia - Ti...,50000,https://blog.ultravoucher.co.id/harga-tiket-bi...,harga tiket bioskop,html_table,2026-04-28 03:19:59
5,UltraVoucher,cinema_ticket,Harga Tiket Bioskop XXI dan CGV Lengkap di Jak...,Jakarta,cinema,None,KOTA KASABLANKA XXI - Tersedia - Tersedia - Ti...,75000,https://blog.ultravoucher.co.id/harga-tiket-bi...,harga tiket bioskop,html_table,2026-04-28 03:19:59
6,UltraVoucher,cinema_ticket,Harga Tiket Bioskop XXI dan CGV Lengkap di Jak...,Jakarta,cinema,None,KOTA KASABLANKA XXI - Tersedia - Tersedia - Ti...,200000,https://blog.ultravoucher.co.id/harga-tiket-bi...,harga tiket bioskop,html_table,2026-04-28 03:19:59
7,UltraVoucher,cinema_ticket,Harga Tiket Bioskop XXI dan CGV Lengkap di Jak...,Jakarta,cinema,None,KOTA KASABLANKA XXI - Tersedia - Tersedia - Ti...,150000,https://blog.ultravoucher.co.id/harga-tiket-bi...,harga tiket bioskop,html_table,2026-04-28 03:19:59
8,UltraVoucher,cinema_ticket,Harga Tiket Bioskop XXI dan CGV Lengkap di Jak...,Jakarta,cinema,None,KOTA KASABLANKA XXI - Tersedia - Tersedia - Ti...,250000,https://blog.ultravoucher.co.id/harga-tiket-bi...,harga tiket bioskop,html_table,2026-04-28 03:19:59
9,UltraVoucher,cinema_ticket,Harga Tiket Bioskop XXI dan CGV Lengkap di Jak...,Jakarta,cinema,None,KUNINGAN CITY XXI - Tersedia - Tidak Tersedia ...,40000,https://blog.ultravoucher.co.id/harga-tiket-bi...,harga tiket bioskop,html_table,2026-04-28 03:19:59


In [8]:
print("Shape:", df_cinema.shape)

display(df_cinema["raw_category"].value_counts())
display(df_cinema["source_platform"].value_counts())
display(df_cinema.sample(min(20, len(df_cinema)), random_state=42))

Shape: (343, 12)


,count
raw_category,
cinema,294
cinema ticket,36
movie snack,9
movie drink,4


,count
source_platform,
UltraVoucher,330
NonaBelanja,13


,source_platform,source_type,restaurant_name,city,raw_category,section,menu_name,price,source_url,search_keyword,extract_method,scraped_at
237,UltraVoucher,cinema_ticket,Harga Tiket Bioskop XXI dan CGV Lengkap di Jak...,Jakarta,cinema,None,Sunter Mall - Tersedia - Tidak Tersedia - Ters...,60000,https://blog.ultravoucher.co.id/harga-tiket-bi...,harga tiket bioskop,html_table,2026-04-28 03:19:59
116,UltraVoucher,cinema_ticket,Harga Tiket Bioskop XXI dan CGV Lengkap di Jak...,Jakarta,cinema,None,CIJANTUNG XXI - Tersedia - Tidak Tersedia - Ti...,45000,https://blog.ultravoucher.co.id/harga-tiket-bi...,harga tiket bioskop,html_table,2026-04-28 03:19:59
113,UltraVoucher,cinema_ticket,Harga Tiket Bioskop XXI dan CGV Lengkap di Jak...,Jakarta,cinema,None,MAL ATRIUM SENEN XXI - Tersedia - Tidak Tersed...,45000,https://blog.ultravoucher.co.id/harga-tiket-bi...,harga tiket bioskop,html_table,2026-04-28 03:19:59
42,UltraVoucher,cinema_ticket,Harga Tiket Bioskop XXI dan CGV Lengkap di Jak...,Jakarta,cinema,None,CITY PLAZA JATINEGARA XXI - Tersedia - Tidak T...,45000,https://blog.ultravoucher.co.id/harga-tiket-bi...,harga tiket bioskop,html_table,2026-04-28 03:19:59
126,UltraVoucher,cinema_ticket,Harga Tiket Bioskop XXI dan CGV Lengkap di Jak...,Jakarta,cinema,None,ONE BELPARK XXI - Tersedia - Tersedia - Tidak ...,100000,https://blog.ultravoucher.co.id/harga-tiket-bi...,harga tiket bioskop,html_table,2026-04-28 03:19:59
275,UltraVoucher,cinema_ticket,Harga Tiket Bioskop XXI dan CGV Lengkap di Jak...,Jakarta,cinema,LOTTE SHOPPING AVENUE XXI,Akhir Pekan:,200000,https://blog.ultravoucher.co.id/harga-tiket-bi...,harga tiket bioskop,text_pattern,2026-04-28 03:19:59
182,UltraVoucher,cinema_ticket,Harga Tiket Bioskop XXI dan CGV Lengkap di Jak...,Jakarta,cinema,None,BAYWALK PLUIT XXI - Tersedia - Tersedia - Tida...,100000,https://blog.ultravoucher.co.id/harga-tiket-bi...,harga tiket bioskop,html_table,2026-04-28 03:19:59
179,UltraVoucher,cinema_ticket,Harga Tiket Bioskop XXI dan CGV Lengkap di Jak...,Jakarta,cinema,None,BAYWALK PLUIT XXI - Tersedia - Tersedia - Tida...,45000,https://blog.ultravoucher.co.id/harga-tiket-bi...,harga tiket bioskop,html_table,2026-04-28 03:19:59
340,NonaBelanja,cinema_concession,Daftar Menu CGV 2026 dan Harganya Terbaru,Unknown,movie drink,Snacks,Soft Drink:,36000,https://nonabelanja.com/menu-cgv/,menu cgv,text_pattern,2026-04-28 03:20:01
25,UltraVoucher,cinema_ticket,Harga Tiket Bioskop XXI dan CGV Lengkap di Jak...,Jakarta,cinema,None,PGC XXI - Tersedia - Tidak Tersedia - Tidak Te...,35000,https://blog.ultravoucher.co.id/harga-tiket-bi...,harga tiket bioskop,html_table,2026-04-28 03:19:59


In [9]:
from google.colab import files

files.download("cinema-scrape.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>